# Phase 5 — frozen legacy training to external MPRA transfer

Run in **Kaggle GPU with Internet ON**. Attach exactly these **three completed Kaggle outputs** as Inputs (a private Kaggle dataset containing the three ZIPs is fine):

1. `[1] enhancer_audit_v1_outputs.zip` — Phase 1 frozen training manifest.
2. `[2.1] enhancer_phase2_corrected_results.zip` — corrected Phase 2 protocol, model revisions, and saved GENA training embeddings.
3. `4_enhancer_phase4_external_dataset.zip` — frozen external MPRA dataset.

Kaggle may expose a ZIP or its extracted members; the loader supports both, validates SHA-256, and stops on ambiguity. No genome reference or raw ENCODE data is required. Existing successful per-model prediction CSVs from a previous Phase 5 Kaggle output can also be attached to resume an interrupted run.

This evaluates **Stage 1 legacy enhancer classifier transfer to relative MPRA activity**, a distinct outcome. It uses all 2,838 frozen Stage 1 training candidates and never fits on the Phase 4 labels. Selected arms are: GC logistic regression, 3-mer logistic regression, spaced 6/4-mer logistic regression, correctly pooled frozen GENA-LM, and fully fine-tuned legacy DNABERT and DNABERT-2 (two epochs each as Phase 2). The raw GC score is a diagnostic. Both HepG2 and K562 are reported separately. The primary head-to-head comparison is DNABERT-2 versus spaced 6/4-mer logistic regression.

**Outputs:** `enhancer_phase5_external_transfer_results.zip` with row-level scores, AUROC/AUPRC, exact-GC comparison, locus bootstrap intervals and a provenance report. Large model weights are saved separately under Kaggle working output (`enhancer_phase5_work/checkpoints`) and are not put into the small results ZIP. Run all cells, then save a Kaggle notebook version to retain checkpoints.


In [1]:
# 0 — Environment and locked run settings.
from pathlib import Path
import os,sys,json,hashlib,zipfile,io,subprocess,random,gc,warnings,shutil,time
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import numpy as np,pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler,Normalizer
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
from itertools import combinations
from sklearn.exceptions import ConvergenceWarning
WORK=Path('/kaggle/working/enhancer_phase5_work');WORK.mkdir(parents=True,exist_ok=True)
PRED=WORK/'predictions';PRED.mkdir(exist_ok=True)
CKPT=WORK/'checkpoints';CKPT.mkdir(exist_ok=True)
HASHES={'phase1_zip':'72aaf60b40304dcf5007f7f4c839ddbf032385e9b7a99fb2d5fc652a3b33fa7c',
'phase2_zip':'332549354d3e41ad0d1e968ad3d4db78ca55beaa47cc3c0a53422ccfec0a6c83',
'phase4_zip':'f5f14a5ad997e9578e5d75732761d9d48d916a11d656a249f807d8c2e33fe1e3',
'stage1_candidate_manifest.csv':'e295155300fafc34528bc1868c6fad6ec78f3912c5b07e034710e81c3b4ac4b3',
'gena_lm_backbone_768_train_and_rc.npz':'6cca1d095f0927d1ff182f28115f233e28dc0041b19be8bc63258e30cce740f3',
'phase4_external_relative_activity_all_cells.csv':'b0bf4a433f1e0af74d95b8747b9312538734f70c425c8379628cafef0c947d2a'}
REVISIONS={'dnabert_legacy':'c56e67ea5827e0ddc67ef059addcf71569b1216e',
'dnabert2':'ec1f874253852eb3907081f57294991b4280ceb6',
'gena_lm':'4f1352bd4e820f1dba341047f54ad7795e083bf2'}
MODEL_IDS={'dnabert_legacy':'zhihan1996/DNA_bert_6',
'dnabert2':'zhihan1996/DNABERT-2-117M',
'gena_lm':'AIRI-Institute/gena-lm-bert-base-t2t'}
CONFIG={'seed':20260928,'stage':1,'reverse_complement_training':True,
 'classical_C':1.0,'classical_max_iter':600,'gena_max_iter':2000,
 'finetune_epochs':2,'finetune_batch':8,'finetune_accum':2,'finetune_lr':2e-5,
 'embedding_batch':32,'inference_batch':32,'max_tokens':256,
 'bootstrap_repeats':300,'models':['gc_logreg','k3_logreg','spaced6_4_logreg',
 'frozen_gena_lm','finetuned_dnabert_legacy','finetuned_dnabert2'],
 'input_hashes':HASHES,'model_revisions':REVISIONS}
def sha(b):return hashlib.sha256(b).hexdigest()
def fsha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
LOCK=sha(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode())
print('Phase 5 protocol SHA256',LOCK)


Phase 5 protocol SHA256 4a1e7bead22829b5ea866bceaf80fc02d174cb18c2385040a764f65eec9bdee4


In [2]:
# 1 — Read exact input members, whether Kaggle supplies ZIPs or extracted outputs.
ROOT=Path('/kaggle/input')
assert ROOT.exists(),'Attach all three required completed Kaggle outputs as Inputs.'
ARCHIVES={
 'phase1':(['[1] enhancer_audit_v1_outputs.zip','enhancer_audit_v1_outputs.zip'],HASHES['phase1_zip']),
 'phase2':(['[2.1] enhancer_phase2_corrected_results.zip','enhancer_phase2_corrected_results.zip'],HASHES['phase2_zip']),
 'phase4':(['4_enhancer_phase4_external_dataset.zip','enhancer_phase4_external_dataset.zip'],HASHES['phase4_zip'])}
ZIP={}
for kind,(names,h) in ARCHIVES.items():
 candidates=[p for p in ROOT.rglob('*') if p.is_file() and p.name in names]
 valid=[p for p in candidates if fsha(p)==h]
 if len(valid)>1:raise RuntimeError(f'Duplicate valid {kind} ZIPs: {valid}')
 if candidates and not valid:raise RuntimeError(f'{kind} ZIP exists but has the wrong SHA256: {candidates}')
 ZIP[kind]=valid[0] if valid else None

def member(kind,name,expected_sha=None):
 if ZIP[kind]:
  with zipfile.ZipFile(ZIP[kind]) as z:
   assert z.testzip() is None,f'Corrupted {kind} ZIP'
   hits=[n for n in z.namelist() if n.split('/')[-1]==name]
   assert len(hits)==1,(kind,name,hits)
   raw=z.read(hits[0])
 else:
  hits=[p for p in ROOT.rglob(name) if p.is_file()]
  valid=[p for p in hits if expected_sha is None or fsha(p)==expected_sha]
  assert len(valid)==1,f'Attach one Phase {kind} input containing {name}. Found valid: {valid}; all: {hits}'
  raw=valid[0].read_bytes()
 if expected_sha:assert sha(raw)==expected_sha,f'{kind}:{name} SHA256 mismatch'
 return raw

manifest=member('phase1','stage1_candidate_manifest.csv',HASHES['stage1_candidate_manifest.csv'])
train_all=pd.read_csv(io.BytesIO(manifest),dtype=str,keep_default_na=False)
assert len(train_all)==3268 and train_all.record_id.is_unique
assert train_all.sequence.str.fullmatch('[ACGT]{200}').all()
train=train_all[train_all.role=='train_candidate'].copy().reset_index(drop=True)
assert len(train)==2838 and set(train.label)=={'0','1'}
assert train.groupby('similarity_group_id').fold_5.nunique().max()==1
assert all(sha(s.encode())==h for s,h in zip(train.sequence,train.sequence_sha256))

correction=json.loads(member('phase2','correction_report.json'))
assert correction['input_manifest_sha256']['stage1_candidate_manifest.csv']==HASHES['stage1_candidate_manifest.csv']
assert correction['model_revisions']=={k:REVISIONS[k] for k in REVISIONS}
assert correction['config_hash']=='63e47ce62c02127cc6bf71762c1205d3bf473cc84085cae3f455ceffb7b1e5a3'
phase4_audit=json.loads(member('phase4','phase4_audit_and_provenance.json'))
assert phase4_audit['output_sha256']['phase4_external_relative_activity_all_cells.csv']==HASHES['phase4_external_relative_activity_all_cells.csv']
assert phase4_audit['protocol_sha256']=='e49fe4d57dda904bffe9fa5a47d8c8e397668cbee71f24e3b0152fd1781b8705'
raw_external=member('phase4','phase4_external_relative_activity_all_cells.csv',HASHES['phase4_external_relative_activity_all_cells.csv'])
external=pd.read_csv(io.BytesIO(raw_external),dtype={'record_id':str,'chrom':str,'design_name':str,'sequence':str,'sequence_sha256':str,'locus_group_id':str})
assert len(external)==30340 and external.record_id.is_unique
assert external.cell_line.value_counts().to_dict()=={'K562':17198,'HepG2':13142}
assert external.sequence.str.fullmatch('[ACGT]{200}').all()
assert all(sha(s.encode())==h for s,h in zip(external.sequence,external.sequence_sha256))
assert set(external.label)=={0,1}
assert external.groupby(['chrom','start','end']).locus_group_id.nunique().max()==1
assert set(train.sequence).isdisjoint(set(external.sequence))
print('VERIFIED training',len(train),'external',len(external),'protocol',phase4_audit['protocol_sha256'])


VERIFIED training 2838 external 30340 protocol e49fe4d57dda904bffe9fa5a47d8c8e397668cbee71f24e3b0152fd1781b8705


In [3]:
# 2 — Common prediction persistence and conventional baselines.
RC=str.maketrans('ACGT','TGCA')
def revcomp(s):return s.translate(RC)[::-1]
def seeds(n):
 random.seed(n);np.random.seed(n)
 if 'torch' in sys.modules:
  torch.manual_seed(n);torch.cuda.manual_seed_all(n)
T=train.sequence.tolist();Y=train.label.astype(int).to_numpy()
E=external.sequence.tolist();IDS=external.record_id.astype(str).tolist()
TA=T+[revcomp(s) for s in T];YA=np.concatenate([Y,Y])
def gc_pair(seqs):return np.array([[(s.count('G')+s.count('C'))/200,s.count('CG')/199] for s in seqs],dtype=float)
def token3(s):return [s[i:i+3] for i in range(198)]
MASKS=tuple(combinations(range(6),4))
def spaced_tokens(s):return [str(mi)+''.join(s[i+j] for j in mask) for i in range(195) for mi,mask in enumerate(MASKS)]
def pred_path(model):return PRED/f'{model}_scores.csv'
def previous(model):
 p=pred_path(model)
 if p.exists():return p
 hits=[q for q in ROOT.rglob(p.name) if q.is_file()]
 if len(hits)>1:raise RuntimeError(f'Duplicate resume scores for {model}: {hits}')
 return hits[0] if hits else None
def checked_scores(model):
 p=previous(model)
 if p is None:return None
 d=pd.read_csv(p,dtype={'record_id':str,'model':str,'protocol_hash':str,'score':float})
 assert len(d)==len(external) and d.record_id.tolist()==IDS
 assert d.model.eq(model).all() and d.protocol_hash.eq(LOCK).all()
 assert np.isfinite(d.score).all() and d.score.between(0,1).all()
 if p!=pred_path(model):shutil.copy2(p,pred_path(model))
 print('REUSED verified',model,flush=True)
 return d.score.to_numpy()
def save_scores(model,values):
 v=np.asarray(values,dtype=float)
 assert v.shape==(len(external),) and np.isfinite(v).all() and np.all((0<=v)&(v<=1))
 p=pred_path(model);tmp=p.with_suffix('.tmp')
 pd.DataFrame({'record_id':IDS,'model':model,'score':v,'protocol_hash':LOCK}).to_csv(tmp,index=False,float_format='%.12g')
 os.replace(tmp,p)
 assert checked_scores(model) is not None
 print('SAVED',model,p.stat().st_size,flush=True)

def classical(model):
 seeds(CONFIG['seed'])
 if model=='gc_logreg':
  clf=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=600))
  clf.fit(gc_pair(TA),YA);return clf.predict_proba(gc_pair(E))[:,1]
 if model=='k3_logreg':
  vec=CountVectorizer(analyzer=token3,lowercase=False,dtype=np.float32)
  clf=make_pipeline(Normalizer(norm='l1'),LogisticRegression(C=1,max_iter=600))
 elif model=='spaced6_4_logreg':
  vec=TfidfVectorizer(analyzer=spaced_tokens,lowercase=False,dtype=np.float32,norm='l2')
  clf=LogisticRegression(C=1,max_iter=600,solver='liblinear')
 else:raise ValueError(model)
 xx=vec.fit_transform(TA)
 with warnings.catch_warnings():
  warnings.filterwarnings('error',category=ConvergenceWarning)
  clf.fit(xx,YA)
 out=[]
 for i in range(0,len(E),512):out.extend(clf.predict_proba(vec.transform(E[i:i+512]))[:,1])
 return np.asarray(out)
for model in ('gc_logreg','k3_logreg','spaced6_4_logreg'):
 if checked_scores(model) is None:save_scores(model,classical(model))


REUSED verified gc_logreg
SAVED gc_logreg 3364648
REUSED verified k3_logreg
SAVED k3_logreg 3364377
REUSED verified spaced6_4_logreg
SAVED spaced6_4_logreg 3576900


In [4]:
# 3 — GPU, pinned model code, and identical tokenization/pooling conventions to Phase 2.
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
assert torch.cuda.is_available(),'Select a Kaggle GPU accelerator.'
DEVICE=torch.device('cuda:0')
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
try:torch.use_deterministic_algorithms(True,warn_only=True)
except Exception:pass
from importlib.metadata import version,PackageNotFoundError
try:installed=version('transformers')
except PackageNotFoundError:installed='missing'
if installed!='4.44.2':
 assert 'transformers' not in sys.modules,'Restart session; Transformers was imported before pinning.'
 subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.44.2'],check=True,timeout=900)
from transformers import AutoModel,AutoTokenizer
import transformers
assert transformers.__version__=='4.44.2'
print('GPU',torch.cuda.get_device_name(0),'Transformers',transformers.__version__)

def text_for(key,s):return ' '.join(s[i:i+6] for i in range(195)) if key=='dnabert_legacy' else s
def tokenize(tok,key,seqs):
 b=tok([text_for(key,s) for s in seqs],padding=True,add_special_tokens=True,truncation=False,return_tensors='pt')
 assert b['input_ids'].shape[1]<=CONFIG['max_tokens']
 return {k:v.to(DEVICE) for k,v in b.items() if k in ('input_ids','attention_mask')}
def pooled(enc,b):
 out=enc(**b)
 h=getattr(out,'last_hidden_state',None)
 if h is None:h=out[0]
 assert h.ndim==3 and h.shape[:2]==b['input_ids'].shape
 mask=b['attention_mask'].to(h.dtype).unsqueeze(-1)
 return (h*mask).sum(1)/mask.sum(1).clamp(min=1)
def encoder(key):
 remote=MODEL_IDS[key];revision=REVISIONS[key]
 tok=AutoTokenizer.from_pretrained(remote,revision=revision,trust_remote_code=True)
 if key=='dnabert_legacy':
  from transformers.models.bert.configuration_bert import BertConfig
  cfg=BertConfig.from_pretrained(remote,revision=revision)
  model=AutoModel.from_pretrained(remote,revision=revision,trust_remote_code=True,config=cfg)
 else:
  try:model=AutoModel.from_pretrained(remote,revision=revision,trust_remote_code=True)
  except (ValueError,TypeError) as err:
   if key!='dnabert2':raise
   from transformers.models.bert.configuration_bert import BertConfig
   cfg=BertConfig.from_pretrained(remote,revision=revision)
   print('Applying pinned DNABERT-2 BertConfig workaround:',type(err).__name__)
   model=AutoModel.from_pretrained(remote,revision=revision,trust_remote_code=True,config=cfg)
 if key=='dnabert2':
  import importlib
  module=importlib.import_module(type(model).__module__)
  assert hasattr(module,'BertUnpadSelfAttention') and hasattr(module,'flash_attn_qkvpacked_func')
  module.flash_attn_qkvpacked_func=None
  print('DNABERT-2 author PyTorch attention fallback active')
 return tok,model.to(DEVICE)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 83.8 MB/s eta 0:00:00
GPU Tesla T4 Transformers 4.44.2


In [5]:
# 4 — Corrected frozen GENA backbone: reuse verified old training embeddings, encode new rows once.
model='frozen_gena_lm'
if checked_scores(model) is None:
 raw=member('phase2','gena_lm_backbone_768_train_and_rc.npz',HASHES['gena_lm_backbone_768_train_and_rc.npz'])
 meta=json.loads(member('phase2','gena_lm_backbone_768_metadata.json'))
 assert meta['dimension']==768 and meta['rows']==2838 and meta['revision']==REVISIONS['gena_lm']
 with np.load(io.BytesIO(raw),allow_pickle=False) as a:
  assert a['record_id'].tolist()==train.record_id.tolist()
  x=np.concatenate([a['original'],a['reverse']]).astype(np.float32)
  assert x.shape==(len(YA),768) and np.isfinite(x).all()
 fit=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=2000,solver='lbfgs'))
 with warnings.catch_warnings():
  warnings.filterwarnings('error',category=ConvergenceWarning)
  fit.fit(x,YA)
 del x,raw;gc.collect()
 tok,enc=encoder('gena_lm')
 assert hasattr(enc,'bert') and enc.config.hidden_size==768
 enc.eval();all_scores=[]
 with torch.inference_mode():
  for start in range(0,len(E),CONFIG['embedding_batch']):
   batch=tokenize(tok,'gena_lm',E[start:start+CONFIG['embedding_batch']])
   h=enc.bert(**batch,return_dict=True).last_hidden_state
   mask=batch['attention_mask'].to(h.dtype).unsqueeze(-1)
   z=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
   assert z.shape[1]==768
   all_scores.extend(fit.predict_proba(z.cpu().numpy().astype(np.float32))[:,1])
   if (start//CONFIG['embedding_batch'])%200==0:print('GENA encoded',start,'/',len(E),flush=True)
 save_scores(model,all_scores)
 del tok,enc,fit,all_scores;gc.collect();torch.cuda.empty_cache()


tokenizer_config.json:   0%|          | 0.00/46.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

GENA encoded 19200 / 30340


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modelin

GENA encoded 25600 / 30340


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modelin

REUSED verified frozen_gena_lm
SAVED frozen_gena_lm 3519187


In [6]:
# 5 — Train full Stage 1 once per fine-tuned arm, save weights, then score both cell lines.
class BinaryEncoder(nn.Module):
 def __init__(self,enc):
  super().__init__();self.encoder=enc
  dim=getattr(enc.config,'hidden_size',None)
  assert isinstance(dim,int) and dim>0
  self.dropout=nn.Dropout(0.1);self.head=nn.Linear(dim,2)
 def forward(self,b):return self.head(self.dropout(pooled(self.encoder,b)))

def train_and_score(key):
 model='finetuned_'+key
 if checked_scores(model) is not None:return
 seeds(CONFIG['seed']+100)
 tok,enc=encoder(key);net=BinaryEncoder(enc).to(DEVICE)
 ckpt=CKPT/f'{model}_stage1_state.pt'
 if ckpt.exists():
  saved=torch.load(ckpt,map_location='cpu',weights_only=True)
  assert saved['lock']==LOCK and saved['revision']==REVISIONS[key]
  net.load_state_dict(saved['state_dict']);print('Loaded verified fitted weights',ckpt)
 else:
  class TrainSet(Dataset):
   def __len__(self):return len(TA)
   def __getitem__(self,i):return TA[i],int(YA[i])
  def collate(items):
   return tokenize(tok,key,[s for s,_ in items]),torch.tensor([y for _,y in items],dtype=torch.long,device=DEVICE)
  ds=TrainSet();opt=torch.optim.AdamW(net.parameters(),lr=CONFIG['finetune_lr'],weight_decay=.01)
  scaler=torch.amp.GradScaler('cuda',enabled=True)
  for epoch in range(CONFIG['finetune_epochs']):
   loader=DataLoader(ds,batch_size=CONFIG['finetune_batch'],shuffle=True,num_workers=0,
     collate_fn=collate,generator=torch.Generator().manual_seed(CONFIG['seed']+100+epoch))
   net.train();opt.zero_grad(set_to_none=True);losses=[]
   for step,(b,y) in enumerate(loader):
    with torch.autocast('cuda',dtype=torch.float16):
     loss=nn.functional.cross_entropy(net(b),y)
    assert torch.isfinite(loss),f'{key}: nonfinite loss'
    scaler.scale(loss/CONFIG['finetune_accum']).backward()
    if (step+1)%CONFIG['finetune_accum']==0 or step+1==len(loader):
     scaler.unscale_(opt);nn.utils.clip_grad_norm_(net.parameters(),1.0)
     scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
    losses.append(loss.item())
   print(key,'epoch',epoch+1,'mean loss',float(np.mean(losses)),flush=True)
  temp=ckpt.with_suffix('.part')
  torch.save({'lock':LOCK,'revision':REVISIONS[key],
              'state_dict':{k:v.cpu() for k,v in net.state_dict().items()}},temp)
  os.replace(temp,ckpt);del opt,scaler,loader,ds
 net.eval();scores=[]
 with torch.inference_mode():
  for start in range(0,len(E),CONFIG['inference_batch']):
   b=tokenize(tok,key,E[start:start+CONFIG['inference_batch']])
   with torch.autocast('cuda',dtype=torch.float16):
    scores.extend(torch.softmax(net(b).float(),dim=1)[:,1].cpu().numpy())
   if (start//CONFIG['inference_batch'])%200==0:print(key,'scored',start,'/',len(E),flush=True)
 save_scores(model,scores)
 del net,enc,tok,scores;gc.collect();torch.cuda.empty_cache()
for key in ('dnabert_legacy','dnabert2'):train_and_score(key)


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

flash_attn_triton.py: 0.00B [00:00, ?B/s]

Applying pinned DNABERT-2 BertConfig workaround: ValueError


pytorch_model.bin:   0%|          | 0.00/468M [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DNABERT-2 author PyTorch attention fallback active
dnabert2 epoch 1 mean loss 0.49454727172851565
dnabert2 epoch 2 mean loss 0.3715541597823022
dnabert2 scored 0 / 30340
dnabert2 scored 6400 / 30340
dnabert2 scored 12800 / 30340
dnabert2 scored 19200 / 30340
dnabert2 scored 25600 / 30340
REUSED verified finetuned_dnabert2
SAVED finetuned_dnabert2 3637189


In [7]:
# 6 — External transfer metrics, raw GC diagnostic and exact-GC pair AUC.
def exact_gc_auc(y,score,g):
 win=0.;total=0
 for count in np.unique(g):
  take=g==count;p=np.sort(score[take & (y==1)]);n=np.sort(score[take & (y==0)])
  if len(p)==0 or len(n)==0:continue
  left=np.searchsorted(n,p,'left');right=np.searchsorted(n,p,'right')
  win+=float((left+.5*(right-left)).sum());total+=len(p)*len(n)
 return win/total if total else np.nan,int(total)

scores={m:checked_scores(m) for m in CONFIG['models']}
assert all(v is not None for v in scores.values())
scores['raw_gc']=np.array([(s.count('G')+s.count('C'))/200 for s in E],dtype=float)
all_rows=[];metrics=[]
for cell in ('HepG2','K562'):
 idx=np.flatnonzero(external.cell_line.to_numpy()==cell)
 y=external.label.to_numpy(dtype=int)[idx]
 counts=np.array([E[i].count('G')+E[i].count('C') for i in idx])
 for model,v in scores.items():
  p=v[idx];auc,pairs=exact_gc_auc(y,p,counts)
  metrics.append({'cell_line':cell,'model':model,'n':len(idx),'positive':int(y.sum()),
    'auroc':float(roc_auc_score(y,p)),'auprc':float(average_precision_score(y,p)),
    'exact_gc_pair_auc':float(auc),'exact_gc_pairs':pairs,
    'spearman_activity':float(spearmanr(p,external.mean_log2_activity.to_numpy()[idx]).statistic)})
 for model,v in scores.items():
  all_rows.append(pd.DataFrame({'record_id':external.record_id.iloc[idx].to_numpy(),
    'cell_line':cell,'locus_group_id':external.locus_group_id.iloc[idx].to_numpy(),
    'label':y,'gc_count':counts,'mean_log2_activity':external.mean_log2_activity.iloc[idx].to_numpy(),
    'model':model,'score':v[idx],'protocol_hash':LOCK}))
M=pd.DataFrame(metrics);print(M.to_string(index=False))
O=pd.concat(all_rows,ignore_index=True)


REUSED verified gc_logreg
REUSED verified k3_logreg
REUSED verified spaced6_4_logreg
REUSED verified frozen_gena_lm
REUSED verified finetuned_dnabert_legacy
REUSED verified finetuned_dnabert2
cell_line                    model     n  positive    auroc    auprc  exact_gc_pair_auc  exact_gc_pairs  spearman_activity
    HepG2                gc_logreg 13142      6683 0.648401 0.666554           0.513204          769085           0.244653
    HepG2                k3_logreg 13142      6683 0.650791 0.656738           0.531615          769085           0.246810
    HepG2         spaced6_4_logreg 13142      6683 0.656667 0.665808           0.554108          769085           0.254189
    HepG2           frozen_gena_lm 13142      6683 0.611582 0.638151           0.511447          769085           0.178754
    HepG2 finetuned_dnabert_legacy 13142      6683 0.640306 0.648467           0.531026          769085           0.225516
    HepG2       finetuned_dnabert2 13142      6683 0.611261 0.584753  

In [8]:
# 7 — Paired locus bootstrap, with a fixed seed and the same resampled loci for all models.
# Repeated overlapping windows and cross-cell loci are resampled as one unit.
BOOT=[];rng=np.random.default_rng(CONFIG['seed']+502)
all_groups=external.locus_group_id.astype(str).unique()
# Per-cell bootstrap avoids failures if one resample lacks a cell or class.
for cell in ('HepG2','K562'):
 idx=np.flatnonzero(external.cell_line.to_numpy()==cell)
 y=external.label.to_numpy(dtype=int)[idx]
 group=external.locus_group_id.astype(str).to_numpy()[idx]
 unique,inv=np.unique(group,return_inverse=True)
 group_rows=[np.flatnonzero(inv==i) for i in range(len(unique))]
 local={m:v[idx] for m,v in scores.items()}
 for b in range(CONFIG['bootstrap_repeats']):
  draw=rng.integers(len(unique),size=len(unique))
  sampled=np.concatenate([group_rows[i] for i in draw])
  yy=y[sampled]
  if len(np.unique(yy))<2:raise ValueError('Bootstrap resample has only one label')
  aucs={m:float(roc_auc_score(yy,v[sampled])) for m,v in local.items()}
  for m,value in aucs.items():
   BOOT.append({'cell_line':cell,'repeat':b,'model':m,'auroc':value,
                'delta_vs_spaced6_4':value-aucs['spaced6_4_logreg']})
 boot=pd.DataFrame(BOOT)
 print(cell,'bootstrap completed',CONFIG['bootstrap_repeats'],flush=True)
boot=pd.DataFrame(BOOT)
intervals=boot.groupby(['cell_line','model']).agg(
 auroc_low=('auroc',lambda x:float(np.quantile(x,.025))),
 auroc_high=('auroc',lambda x:float(np.quantile(x,.975))),
 delta_low=('delta_vs_spaced6_4',lambda x:float(np.quantile(x,.025))),
 delta_high=('delta_vs_spaced6_4',lambda x:float(np.quantile(x,.975)))).reset_index()
M=M.merge(intervals,on=['cell_line','model'],validate='one_to_one')


HepG2 bootstrap completed 300
K562 bootstrap completed 300


In [9]:
# 8 — Package small auditable results. Fitted weights remain in Kaggle working Outputs.
ART=WORK/'results';ART.mkdir(exist_ok=True)
outputs={}
def csvout(name,frame):
 p=ART/name;frame.to_csv(p,index=False,float_format='%.12g');outputs[name]=p
csvout('phase5_external_predictions.csv',O)
csvout('phase5_external_metrics.csv',M)
csvout('phase5_locus_bootstrap.csv',boot)
report={'protocol_hash':LOCK,'configuration':CONFIG,'source_sha256':HASHES,
 'phase2_config_hash':correction['config_hash'],
 'phase4_protocol_hash':phase4_audit['protocol_sha256'],
 'interpretation':'Legacy Stage 1 enhancer classification transferred to top/bottom MPRA activity; outcomes differ.',
 'primary_comparison':'finetuned_dnabert2 vs spaced6_4_logreg; descriptive transfer only',
 'significance':'Phase 4 source BED has -1 P/Q placeholders; no statistical assay significance asserted.',
 'model_weight_paths':{key:str(CKPT/f'finetuned_{key}_stage1_state.pt') for key in ('dnabert_legacy','dnabert2')},
 'output_sha256':{name:fsha(path) for name,path in outputs.items()},
 'software':{'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,
             'torch':torch.__version__,'transformers':transformers.__version__}}
report_path=ART/'phase5_provenance.json';report_path.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
outputs[report_path.name]=report_path
archive=Path('/kaggle/working/enhancer_phase5_external_transfer_results.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=9) as z:
 for name,p in outputs.items():z.write(p,name)
with zipfile.ZipFile(archive) as z:
 assert z.testzip() is None
 for name,h in report['output_sha256'].items():assert sha(z.read(name))==h
print('DOWNLOAD',archive,'bytes',archive.stat().st_size)
print('Checkpoints saved under',CKPT,'(retain Kaggle Notebook Outputs for reproducibility).')


DOWNLOAD /kaggle/working/enhancer_phase5_external_transfer_results.zip bytes 6201198
Checkpoints saved under /kaggle/working/enhancer_phase5_work/checkpoints (retain Kaggle Notebook Outputs for reproducibility).
